In [9]:
import os
import duckdb
import pandas as pd
import numpy as np
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [10]:
load_dotenv(dotenv_path="../.env")
minio_user = os.getenv("MINIO_ROOT_USER", "minioadmin")
minio_password = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE SECRET IF NOT EXISTS (
        TYPE S3,
        KEY_ID '{minio_user}',
        SECRET '{minio_password}',
        ENDPOINT 'localhost:9000',
        URL_STYLE 'path',
        USE_SSL false
    );
""")

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Financial Predictive Models vers2")

print(" Connected to DuckDB Data Lake and MLflow Dashboard successfully.")


 Connected to DuckDB Data Lake and MLflow Dashboard successfully.


In [11]:
print("Loading Data Lake into memory...")
df = con.execute("SELECT * FROM read_parquet('s3://analytics-data/ml_features.parquet')").df()

print("Loading Quality Assurance metrics...")
stats_df = con.execute("SELECT * FROM read_parquet('s3://analytics-data/feature_statistics.parquet')").df()

best_features_df = stats_df[stats_df['quality_flag'] == 'PASS'].nlargest(15, 'importance_score')
ml_features_list = best_features_df['feature_name'].tolist()

print("\n--- DATA LOADED ---")
print(f"Total Rows: {df.shape[0]:,}")
print(f"Top 15 ML Features: {', '.join(ml_features_list)}")


Loading Data Lake into memory...


Loading Quality Assurance metrics...

--- DATA LOADED ---
Total Rows: 514,971
Top 15 ML Features: hl_ratio, close_position, bb_percentage, stoch_k, roc_10, roc_20, bb_width, stoch_d, rsi_14, volume_ratio, obv, macd_histogram, volume, prev_volume, macd


In [12]:
def prepare_asset_data(df, symbol, timeframe, features, test_start_date):
    """
    Filters data for a specific asset, drops NaN values, 
    and splits into Train/Test based on a specific date.
    """
    clean_data = df[(df['asset_symbol'] == symbol) & (df['interval'] == timeframe)].copy()
    clean_data = clean_data.dropna(subset=features + ['returns_1d'])
    
    clean_data = clean_data.sort_values('timestamp')
    
    if len(clean_data) < 100:
        print(f"Warning: Not enough data for {symbol} on {timeframe}.")
        return None, None, None, None
    
    train_data = clean_data[clean_data['timestamp'] < test_start_date]
    test_data = clean_data[clean_data['timestamp'] >= test_start_date]
    
    print(f"Data Split for {symbol}:")
    print(f"  Training Rows: {len(train_data)}")
    print(f"  Testing Rows:  {len(test_data)}\n")
    
    X_train = train_data[features]
    y_train = np.where(train_data['returns_1d'] > 0, 1, 0)
    
    X_test = test_data[features]
    y_test = np.where(test_data['returns_1d'] > 0, 1, 0)
    
    return X_train, X_test, y_train, y_test
